# 02 — Candidate Generation

Generate a high-recall candidate set without all-vs-all comparison.

In [ ]:
from pathlib import Path
import pandas as pd
import sys
ROOT = Path.cwd().resolve()
if not (ROOT/'data').exists() and (ROOT.parent/'data').exists(): ROOT=ROOT.parent
sys.path.insert(0, str(ROOT/'src'))
from preprocessing import prepare_source
TRAIN=ROOT/'data'/'train'
s1=prepare_source(pd.read_csv(TRAIN/'train_source1.tsv', sep='\t'))
s2=prepare_source(pd.read_csv(TRAIN/'train_source2.tsv', sep='\t'))
s3=prepare_source(pd.read_csv(TRAIN/'train_source3.tsv', sep='\t'))

In [ ]:
for df in (s1,s2,s3):
    df['name_country_key'] = df['name_norm'] + '||' + df['country_norm']
    df['address_block'] = df['address_norm'].str.replace(' ','',regex=False).str[:8]
    df['address_country_key'] = df['address_block'] + '||' + df['country_norm']

In [ ]:
def block_pairs(s1, other):
    left=s1[['entity_id','name_country_key','address_country_key']].rename(columns={'entity_id':'source1_entity_id'})
    right=other[['entity_id','name_country_key','address_country_key']].rename(columns={'entity_id':'candidate_entity_id'})
    by_name=left.merge(right,on='name_country_key',how='inner')[['source1_entity_id','candidate_entity_id']]
    by_addr=left.merge(right,on='address_country_key',how='inner')[['source1_entity_id','candidate_entity_id']]
    out=pd.concat([by_name,by_addr],ignore_index=True).drop_duplicates()
    return out

c2=block_pairs(s1,s2); c3=block_pairs(s1,s3)
candidates=pd.concat([c2.assign(source='S2'),c3.assign(source='S3')],ignore_index=True).drop_duplicates(['source1_entity_id','candidate_entity_id'])
print('Candidate pairs:', candidates.shape)

In [ ]:
# Required submission-format candidate file: one row per S1 entity.
candidate_output=(candidates.groupby('source1_entity_id')['candidate_entity_id']
                   .apply(lambda x: ','.join(sorted(set(x))))
                   .reindex(s1['entity_id'], fill_value='')
                   .rename('candidate_entity_ids').reset_index())
(candidate_output).to_csv(ROOT/'output'/'candidate_pairs.tsv',sep='\t',index=False)
display(candidate_output.head())

In [ ]:
# Keep the pair-level candidate table for the next notebook.
candidates.to_pickle(ROOT/'output'/'candidate_pairs_internal.pkl')
print('Saved output/candidate_pairs.tsv and output/candidate_pairs_internal.pkl')